# astrowidget — Interactive Sky Viewer

Replaces ipyaladin for OVRO-LWA visualization. No FITS round-trip — zarr → numpy → GPU directly.

**What this replaces:**
| ipyaladin workflow | astrowidget workflow |
|---|---|
| Extract WCS from zarr attrs | Same |
| Build `astropy.io.fits.HDUList` from numpy | **Eliminated** |
| Serialize FITS to bytes | **Eliminated** |
| `aladin.add_fits(hdul)` | `widget.set_image(array, wcs)` |
| Broken sphere overlay | Working sphere overlay |

## Setup

Run this notebook from the `ovro-lwa-portal` pixi environment with astrowidget installed:
```
pip install -e /path/to/astrowidget
```

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

# Load OSN credentials from .env
env_path = Path.cwd().parent / ".env"
for line in env_path.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        key, _, val = line.partition("=")
        os.environ.setdefault(key.strip(), val.strip().strip('"').strip("'"))

import ovro_lwa_portal as ovro

## 1. Load OVRO-LWA data from OSN

Same `open_dataset()` call as the interactive_viz_demo notebook.

In [ ]:
# DOI for the 10t x 10f x 4096x4096 test dataset
DOI = "10.33569/4q7nb-ahq31"

ds = ovro.open_dataset(
    DOI,
    production=False,
    storage_options={
        "key": os.environ["OSN_KEY"],
        "secret": os.environ["OSN_SECRET"],
    },
)

print(f"Loaded: {DOI}")
print(f"  Dims:  {dict(ds.dims)}")
print(f"  Vars:  {list(ds.data_vars)}")
print(f"  Freq:  {float(ds.frequency.values[0])/1e6:.1f} – {float(ds.frequency.values[-1])/1e6:.1f} MHz")
print(f"  WCS:   {ds.radport.has_wcs}")

## 2. Extract image + WCS — no FITS construction

This is the key difference from ipyaladin. Instead of:
```python
# OLD: ipyaladin (broken + slow)
hdul = _build_fits_hdu(ds, time_idx=0, freq_idx=0)  # builds FITS from numpy
aladin.add_fits(hdul)                                 # serializes FITS again
```

We go directly:
```python
# NEW: astrowidget (works + fast)
widget.set_image(array, wcs)  # raw float32 bytes, no FITS
```

In [ ]:
import numpy as np
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u

# Extract WCS from zarr metadata (same as ovro-lwa-portal accessor)
wcs = ds.radport._get_wcs()
print(f"Projection: {wcs.wcs.ctype[0]}, {wcs.wcs.ctype[1]}")
print(f"Phase center: RA={wcs.wcs.crval[0]:.4f}°, Dec={wcs.wcs.crval[1]:.4f}°")
print(f"Pixel scale: {abs(wcs.wcs.cdelt[0])*3600:.1f}\"/pixel")

# Load one slice — this is the only S3 read needed
# Downsample to 512x512 for interactive display
STRIDE = max(1, ds.sizes["l"] // 512)
image = ds.SKY.isel(time=0, frequency=0, polarization=0).values[::STRIDE, ::STRIDE]
image = image.astype(np.float32)
print(f"\nImage: {image.shape} (strided {STRIDE}x from {ds.sizes['l']}x{ds.sizes['m']})")

# Adjust WCS for downsampled pixel scale
wcs_display = wcs.deepcopy()
wcs_display.wcs.cdelt = [wcs.wcs.cdelt[0] * STRIDE, wcs.wcs.cdelt[1] * STRIDE]
wcs_display.wcs.crpix = [(wcs.wcs.crpix[0] - 0.5) / STRIDE + 0.5,
                          (wcs.wcs.crpix[1] - 0.5) / STRIDE + 0.5]

## 3. Display on the celestial sphere

`set_image()` sends raw float32 bytes to the GPU. No FITS construction, no serialization.
The SIN projection fragment shader renders the image on the sphere per-pixel at 60fps.

**Controls:**
- **Drag** to rotate the sphere
- **Scroll** to zoom in/out
- **Hover** to see RA/Dec coordinates

In [ ]:
from astrowidget import SkyWidget

widget = SkyWidget()

# Send image directly — zarr → numpy → GPU. No FITS anywhere.
widget.set_image(image, wcs_display)

# Navigate to the phase center with full hemisphere view
phase_center = SkyCoord(
    ra=wcs.wcs.crval[0], dec=wcs.wcs.crval[1],
    unit="deg", frame="fk5",
)
widget.goto(phase_center, fov=180 * u.deg)

print(f"vmin={widget.vmin:.2f}, vmax={widget.vmax:.2f} (auto-scaled 2nd/98th percentile)")
widget

## 4. Navigate to known radio sources

Use `goto()` with any `SkyCoord` to navigate the sphere.

In [ ]:
# Navigate to Cassiopeia A — brightest radio source in the northern sky
cas_a = SkyCoord.from_name("Cas A")
widget.goto(cas_a, fov=30 * u.deg)
print(f"Navigated to Cas A: {cas_a.to_string('hmsdms')}")

In [ ]:
# Navigate to Cygnus A
cyg_a = SkyCoord.from_name("Cyg A")
widget.goto(cyg_a, fov=20 * u.deg)
print(f"Navigated to Cyg A: {cyg_a.to_string('hmsdms')}")

In [ ]:
# Back to full hemisphere
widget.goto(phase_center, fov=180 * u.deg)

## 5. Change display settings from Python

Colormap, stretch, and color scale are GPU-only updates — instant, no data re-upload.

In [ ]:
# Switch colormap (GPU-only — no Python round-trip)
widget.colormap = "viridis"

# Switch stretch function
widget.stretch = "sqrt"

# Adjust color scale manually
widget.auto_scale(percentile_low=5, percentile_high=95)

In [ ]:
# Load a different time/frequency slice — one S3 read, then instant display
new_image = ds.SKY.isel(time=5, frequency=5, polarization=0).values[::STRIDE, ::STRIDE]
new_image = new_image.astype(np.float32)
widget.set_image(new_image, wcs_display)
widget.colormap = "inferno"
widget.stretch = "linear"
print("Loaded time=5, freq=5 slice")

## 6. Read view state from Python

After panning/zooming in the widget, read back the current view position.

In [ ]:
# Read current view (updates after pan/zoom interaction)
print(f"View center: RA={widget.view_ra:.4f}°, Dec={widget.view_dec:.4f}°")
print(f"Field of view: {widget.view_fov:.2f}°")
print(f"Colormap: {widget.colormap}, Stretch: {widget.stretch}")
print(f"Color scale: [{widget.vmin:.2f}, {widget.vmax:.2f}]")